# Variational Autoencoders — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normal_pdf(x,mu,sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2)/(sigma*np.sqrt(2*np.pi))
def softmax(a):
    a=np.asarray(a,dtype=float); e=np.exp(a-a.max()); return e/e.sum()
def mse(a,b): return float(np.mean((np.asarray(a)-np.asarray(b))**2))
print('setup ok')

## the evidence lower bound
Autoencoders (10.1) provide the encoder-decoder skeleton, probability gives Gaussian latents and expectations, and KL divergence measures how far the encoder distribution drifts from a simple prior. The result enables controllable sampling, beta-VAE (10.3), and the latent spaces used by diffusion systems (10.14).

In [ ]:
# 1) reparameterization creates z from mu, sigma, and epsilon
mu=0.5; logv=-0.7; sigma=math.exp(0.5*logv); eps=np.linspace(-2,2,80); z=mu+sigma*eps
plt.figure(figsize=(5,3)); plt.plot(eps,z); plt.xlabel('epsilon'); plt.ylabel('z'); plt.title('z = mu + sigma epsilon')
assert abs(sigma-0.7046880897)<1e-9

In [ ]:
# 2) KL grows as the posterior mean leaves the unit prior
mus=np.linspace(-2,2,100); kl=0.5*(mus**2+math.exp(logv)-1-logv)
plt.figure(figsize=(5,3)); plt.plot(mus,kl); plt.xlabel('mu'); plt.ylabel('KL'); plt.title('mean far from prior costs KL')
assert kl[0]>kl[len(kl)//2]

In [ ]:
# 3) KL also grows when variance is too small or too large
logvs=np.linspace(-3,2,100); klv=0.5*(mu**2+np.exp(logvs)-1-logvs)
plt.figure(figsize=(5,3)); plt.plot(logvs,klv); plt.xlabel('log variance'); plt.ylabel('KL'); plt.title('variance mismatch costs KL')
assert min(klv)<max(klv)

In [ ]:
# 4) decoding nearby latent points gives a smooth curve
z=np.linspace(-2,2,80); x1=np.tanh(z); x2=np.sin(z)
plt.figure(figsize=(4,4)); plt.plot(x1,x2); plt.title('smooth latent path decodes smoothly')
assert np.max(np.abs(np.diff(x1)))<0.1

In [ ]:
# 5) ELBO tradeoff: higher KL can buy lower reconstruction loss
recon=np.array([0.9,0.55,0.42,0.38]); kl=np.array([0.02,0.10,0.2482926519,0.6]); loss=recon+kl
plt.figure(figsize=(5,3)); plt.plot(recon,label='recon'); plt.plot(kl,label='KL'); plt.plot(loss,label='total'); plt.legend(); plt.title('VAE objective balances two pressures')
assert int(np.argmin(loss))==1